# S11 - BI/ML distribuido con Spark

**Objetivo:** calcular, sobre datos reales ya persistidos por S9 y S10, los KPIs del flujo de eventos
(calidad y latencia) y el error de la prediccion de series de tiempo **en produccion** (no el de prueba
offline de S10), y publicarlos en Prometheus via Pushgateway para visualizarlos en el mismo Grafana de S9.

**Alcance:** no se reentrena ningun modelo ni se vuelve a leer Kafka en streaming - este notebook es un
job batch de BI/ML que lee los artefactos ya guardados (Parquet de S9, Parquet de S10) y publica metricas.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("sesion11-bi-ml-distribuido")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/16 21:48:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Cargar los dos artefactos reales de S9 y S10

`RUTA_EVENTOS_S9` es el mismo Parquet que S9 (3.8) dejo escrito con los campos de observabilidad
(`isValid`, `processedAt`, `latencyMs`). `RUTA_PREDICCIONES_S10` es el que S10 (3.7.2) dejo con cada
prediccion en tiempo real (`CE` real del momento, `prediccionSiguienteCE`). Ninguno de los dos se
recalcula hoy - son evidencia ya persistida de sesiones anteriores.

In [30]:
RUTA_EVENTOS_S9 = "../s09-observabilidad-pipeline/artifacts/atmos_eventos_observabilidad"
RUTA_PREDICCIONES_S10 = "../s10-series-tiempo-inferencia-streaming/artifacts/predicciones_stream"

eventos = spark.read.parquet(RUTA_EVENTOS_S9)
predicciones = spark.read.parquet(RUTA_PREDICCIONES_S10)

print(f"Filas eventos (S9): {eventos.count():,}")
print(f"Filas predicciones (S10): {predicciones.count():,}")

Filas eventos (S9): 1,080
Filas predicciones (S10): 6


## 2. KPI del flujo de eventos: volumen, calidad y latencia

Tres numeros que resumen la salud del flujo de eventos IoT/empresariales como KPI de negocio -
distinto del *lag* o la disponibilidad de Kafka que S9 ya mostraba en su propio tablero (eso es
infraestructura; esto es calidad del dato que le llega al negocio).

In [31]:
kpi_eventos = eventos.agg(
    F.count("*").alias("total"),
    F.avg(F.col("isValid").cast("int")).alias("ratioValidos"),
    F.avg("latencyMs").alias("avgLatencyMs"),
).first()

print(f"Eventos procesados:   {kpi_eventos['total']}")
print(f"Ratio de validos:     {kpi_eventos['ratioValidos']:.4f}")
print(f"Latencia promedio:    {kpi_eventos['avgLatencyMs']:.2f} ms")

Eventos procesados:   1080
Ratio de validos:     1.0000
Latencia promedio:    686465.43 ms


## 3. KPI de la prediccion en produccion: error real, no el de prueba

S10 (3.5.1) ya midio RMSE/MAE/R2 sobre un conjunto de prueba historico, antes de desplegar el modelo.
Esa metrica responde "que tan bueno es el modelo en general?". La pregunta de hoy es distinta:
"que tan bien predijo el modelo, de verdad, en las lecturas que ya llegaron en produccion?"

Cada fila de `predicciones_stream` guarda la prediccion del minuto siguiente hecha en el batch `N`.
El valor real de ese minuto siguiente es, precisamente, el `CE` que trae el batch `N+1` de la misma
estacion - no hay que esperar nada nuevo, ya esta en la misma tabla.

In [32]:
actual_siguiente = predicciones.select(
    (F.col("batchId") - 1).alias("batchId"),
    "estacionId",
    F.col("CE").alias("CE_real_siguiente"),
)

comparado = (
    predicciones.alias("p")
    .join(actual_siguiente.alias("a"), ["batchId", "estacionId"], "inner")
    .select(
        "p.batchId", "p.estacionId", "p.CE",
        "p.prediccionSiguienteCE", "a.CE_real_siguiente",
    )
    .withColumn("errorAbsoluto", F.abs(F.col("prediccionSiguienteCE") - F.col("CE_real_siguiente")))
)
comparado.orderBy("batchId").show(truncate=False)

kpi_prediccion = comparado.agg(
    F.count("*").alias("totalComparado"),
    F.avg("errorAbsoluto").alias("maeProduccion"),
).first()

ultima = predicciones.orderBy(F.desc("batchId")).first()

print(f"Predicciones comparables: {kpi_prediccion['totalComparado']}")
print(f"MAE en produccion:        {kpi_prediccion['maeProduccion']:.4f}")
print(f"Ultimo CE real:           {ultima['CE']}")
print(f"Ultima prediccion:        {ultima['prediccionSiguienteCE']:.4f}")

+-------+--------------+-----+---------------------+-----------------+--------------------+
|batchId|estacionId    |CE   |prediccionSiguienteCE|CE_real_siguiente|errorAbsoluto       |
+-------+--------------+-----+---------------------+-----------------+--------------------+
|1      |estacion-ns-01|-5.71|-5.622251572735802   |-5.68            |0.05774842726419749 |
|2      |estacion-ns-01|-5.68|-5.593095284423376   |-5.64            |0.046904715576623346|
|3      |estacion-ns-01|-5.64|-5.554164091685281   |-5.58            |0.02583590831471927 |
|4      |estacion-ns-01|-5.58|-5.4950825544956965  |-5.56            |0.06491744550430312 |
|5      |estacion-ns-01|-5.56|-5.475606506524762   |-5.49            |0.01439349347523855 |
+-------+--------------+-----+---------------------+-----------------+--------------------+

Predicciones comparables: 5
MAE en produccion:        0.0420
Ultimo CE real:           -5.49
Ultima prediccion:        -5.4074


El ultimo batch (el mas reciente) todavia no tiene un batch siguiente con el cual compararse -
por eso `comparado` siempre tiene una fila menos que `predicciones`. Esto no es un error: es la misma
razon por la que la ultima fila del Gold de S04 se quedo sin `label` en S10 (2.2) - la fila mas
reciente, por definicion, todavia no vivio su propio futuro.

## 4. Publicar los KPIs en Prometheus, via Pushgateway

Ninguna de estas metricas las puede exponer un *exporter* como `kafka-exporter` (S9, 2.3): no describen
el estado de un servicio corriendo, son el resultado de un job batch que corre una vez y termina. Para
ese caso exacto existe el **Pushgateway**: un intermediario al que un job de corta duracion *empuja*
sus metricas (en vez de que Prometheus las vaya a buscar), y que las retiene para que Prometheus las
recoja en su siguiente *scrape* normal (Prometheus Authors, 2024b).

In [35]:
import urllib.request

metricas = {
    "lambda26_bi_eventos_total": kpi_eventos["total"],
    "lambda26_bi_eventos_validos_ratio": kpi_eventos["ratioValidos"],
    "lambda26_bi_latencia_promedio_ms": kpi_eventos["avgLatencyMs"],
    "lambda26_bi_prediccion_mae": kpi_prediccion["maeProduccion"],
    "lambda26_bi_prediccion_ultima_real": ultima["CE"],
    "lambda26_bi_prediccion_ultima_predicha": ultima["prediccionSiguienteCE"],
}

lineas = []
for nombre, valor in metricas.items():
    lineas.append(f"# TYPE {nombre} gauge")
    lineas.append(f"{nombre} {valor}")
cuerpo = "\n".join(lineas) + "\n"

PUSHGATEWAY_URL = "http://pushgateway:9091/metrics/job/lambda26_bi/instance/s11_notebook"
peticion = urllib.request.Request(PUSHGATEWAY_URL, data=cuerpo.encode("utf-8"), method="PUT")
with urllib.request.urlopen(peticion) as respuesta:
    print("Pushgateway respondio:", respuesta.status)

Pushgateway respondio: 200


**Por que `PUT` y no `POST`.** `POST` agrega o reemplaza solo las metricas que el cuerpo incluye,
dejando intactas otras metricas previas del mismo grupo (`job`/`instance`). `PUT` reemplaza **todo** el
grupo por lo que llega en este *push* - correcto aqui porque cada corrida de este notebook calcula el
estado completo y actual de los 6 KPIs, no un incremento parcial (Prometheus Authors, 2024b).

## 5. Verificar que Prometheus ya las tiene

Antes de armar el tablero de Grafana (seccion 3.5 de la guia), confirma en PromQL que las 6 metricas
ya responden con un valor - si Grafana muestra "No data", el problema esta aqui, no en el panel.

In [36]:
import json

for nombre in metricas:
    url = f"http://prometheus:9090/api/v1/query?query={nombre}"
    resultado = json.load(urllib.request.urlopen(url))
    valores = resultado["data"]["result"]
    if valores:
        print(f"{nombre}: {valores[0]['value'][1]}")
    else:
        print(f"{nombre}: SIN DATOS (revisa 4.1)")

lambda26_bi_eventos_total: 1080
lambda26_bi_eventos_validos_ratio: 1
lambda26_bi_latencia_promedio_ms: 686465.4314814815
lambda26_bi_prediccion_mae: 0.04195999802701635
lambda26_bi_prediccion_ultima_real: -5.49
lambda26_bi_prediccion_ultima_predicha: -5.4074485969876


**Evidencia de aprendizaje de este notebook:**

- KPI de calidad y latencia del flujo de eventos, calculado sobre la evidencia real de S9.
- Error de prediccion en produccion (no el de prueba offline de S10), calculado con el propio
  historial de `predicciones_stream`.
- Las 6 metricas publicadas en Pushgateway y confirmadas en Prometheus con PromQL real.

Con esto, la seccion 3.5 de la guia arma el tablero de Grafana que visualiza estas 6 metricas.